# Object-summary batch processing

Summarize every Markdown file under these two corpora, in order:

- `PROJ_ROOT/assets/md.json.semantics.coding.4.0`
- `PROJ_ROOT/assets/md.json.semantics.coding.5.0`

Uses [core.dom.summary.object](01.core.dom.summary.object.ipynb) through
`utils.summary_object_asyncio.summarize_object_documents`. Parsing, row summaries,
and image filtering use [core.dom.summary.object_utils](01.core.dom.summary.object_utils.ipynb).

Original figure, table, and list subtrees and captions stay in the recursive DOM.
Markdown pipe tables and HTML tables include a summary for every physical row,
including headers. HTML merged groups nest recursively through `children` and
matching `subnodes`, with summaries at every level and inherited cell context.
Sections, prose, and document summaries remain in the full tree.

Images matching `PROJ_ROOT/assets/images_ignored` remain in the tree but contribute
no summary. Fully ignored figures have `summary_ignored=True` and an empty summary;
mixed figures summarize only retained images. Successful documents atomically
replace their adjacent `<markdown_stem>_semantics.json`.

Run cells in order to start processing. Opening this notebook does not run the
corpora. Embedding and ingestion are excluded.

## Configuration

Use `OLLAMA_HOST`, `OLLAMA_SUMMARY_MODEL`, and optional `OLLAMA_API_KEY` from
`PROJ_ROOT/.env`. The same image-capable model handles text and figures on both
`http://127.0.0.1:11434` and the configured `OLLAMA_HOST` (currently
`http://172.27.74.16:11434`). Individual requests are dispatched asynchronously
to whichever server has an available slot, including requests from one document.
The default shared limit of eight requests is split into four slots per server.
Override the comma-separated server list with `RIBOSOME_OLLAMA_HOSTS` and the
per-server limit with `RIBOSOME_OLLAMA_SERVER_CONCURRENCY`. The shared limit still
bounds total work, including local image decoding. Faster servers receive more
work as they free slots. Both servers must have `OLLAMA_SUMMARY_MODEL` installed.
The notebook does not modify `.env`.

Set limits in the environment or `.env` before running this cell. Defaults are
**4 document workers, 8 shared model requests, and 32 traversal tasks per document**.
Limits are applied before the object summarizer is imported. If it is already
loaded with different limits, restart the kernel and run cells in order.
Markdown is parsed directly; existing AST artifacts are left untouched.

Place static reference images under `PROJ_ROOT/assets/images_ignored`; subfolders
are included. A missing or empty folder disables filtering. Matching uses the
core utility's conservative aspect-ratio, difference-hash, and pixel thresholds,
comparing transparency on both black and white backgrounds. Remote, unsupported,
and animated images follow the normal provider path. Image decoding runs outside
the event loop and shares the concurrency budget with model requests.

`PREVIEW_OUTPUTS` limits the number of saved trees inspected after the batch;
it does not limit processing of the corpora.
Requests disable thinking, cap generated responses at 2,048 tokens, and have a
300-second deadline (including server queue time). Override with
`RIBOSOME_SUMMARY_MAX_TOKENS` and `RIBOSOME_SUMMARY_REQUEST_TIMEOUT` before setup.


In [ ]:
import asyncio
import json
import os
import sys
from pathlib import Path
from time import perf_counter

from dotenv import dotenv_values, load_dotenv


def resolve_project_root():
    configured = os.getenv("PROJ_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / "pyproject.toml").is_file():
            raise FileNotFoundError(f"Invalid PROJ_ROOT (missing pyproject.toml): {root}")
        return root
    cwd = Path.cwd().resolve()
    for root in (cwd, *cwd.parents):
        if (root / "pyproject.toml").is_file():
            return root
    raise FileNotFoundError("Cannot locate PROJ_ROOT containing pyproject.toml")


PROJ_ROOT = resolve_project_root()
ENV_FILE = PROJ_ROOT / ".env"
if not ENV_FILE.is_file():
    raise FileNotFoundError(f"Missing configuration: {ENV_FILE}")
load_dotenv(ENV_FILE, override=False)
env_config = dotenv_values(ENV_FILE)
OLLAMA_HOST = (env_config.get("OLLAMA_HOST") or "").strip()
SUMMARY_MODEL = (env_config.get("OLLAMA_SUMMARY_MODEL") or "").strip()
if not OLLAMA_HOST or not SUMMARY_MODEL:
    raise ValueError("Set OLLAMA_HOST and OLLAMA_SUMMARY_MODEL in PROJ_ROOT/.env")
os.environ["PROJ_ROOT"] = str(PROJ_ROOT)
os.environ["OLLAMA_HOST"] = OLLAMA_HOST
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))

from utils.summary_asyncio import configure_summary_limits
from utils.summary_ollama_pool import OllamaSummaryPool
SUMMARY_LIMITS = configure_summary_limits(summary_module="object")
DOCUMENT_CONCURRENCY = SUMMARY_LIMITS["RIBOSOME_DOCUMENT_CONCURRENCY"]

import pandas as pd
from IPython.display import display
from utils.summary_object_asyncio import discover_markdown, summarize_object_documents
from ribosome.core.dom.summary import object_utils

INPUT_DIRS = [
    PROJ_ROOT / "assets" / "code-docs.4.0" / ".md",
    PROJ_ROOT / "assets" / "code-docs.5.0" / ".md",
]
CHECKPOINT_NAME = ".object-summary-checkpoint.json"
RESUME = True  # Fill missing summaries in saved outputs; False only for an explicit full rebuild.
IGNORED_IMAGES_DIR = PROJ_ROOT / "assets" / "images_ignored"
PREVIEW_OUTPUTS = 3
LANG = "zh"
LEAF_MIN_LEN = 100
MIN_LEN = None
api_key = env_config.get("OLLAMA_API_KEY")
headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}
REQUEST_TIMEOUT = float(os.getenv("RIBOSOME_SUMMARY_REQUEST_TIMEOUT", "300"))
MAX_TOKENS = int(os.getenv("RIBOSOME_SUMMARY_MAX_TOKENS", "2048"))
# Override with a comma-separated list in the environment or .env if needed.
OLLAMA_HOSTS = list(dict.fromkeys(
    host.strip().rstrip("/") for host in os.getenv(
        "RIBOSOME_OLLAMA_HOSTS", f"http://127.0.0.1:11434,{OLLAMA_HOST}"
    ).split(",") if host.strip()
))
SERVER_CONCURRENCY = int(os.getenv(
    "RIBOSOME_OLLAMA_SERVER_CONCURRENCY",
    str(max(1, SUMMARY_LIMITS["RIBOSOME_SUMMARY_CONCURRENCY"] // len(OLLAMA_HOSTS)))
)) if OLLAMA_HOSTS else 0
ollama_client = OllamaSummaryPool(
    hosts=OLLAMA_HOSTS, concurrency_per_server=SERVER_CONCURRENCY,
    headers=headers, request_timeout=REQUEST_TIMEOUT,
    max_tokens=MAX_TOKENS, think=False,
)
print("Inputs:")
for input_dir in INPUT_DIRS:
    print(f"  {input_dir}")
print(f"Ollama servers: {OLLAMA_HOSTS}\nText and image model: {SUMMARY_MODEL}")
print(f"Maximum requests per server: {SERVER_CONCURRENCY}")
print(f"Effective concurrency: {SUMMARY_LIMITS}")
print(f"Ignored-image references: {IGNORED_IMAGES_DIR}")

## Discovery and provider preflight

Recursively discover sorted `.md` files in both corpora, including nested paths
and uppercase `.MD` extensions. Missing or empty corpus directories stop preflight.
All documents are included even if the same manual exists in both versions.

Preview the usable ignored-image reference count, matching settings, and reference
signature. Unreadable references warn once and are skipped. Each corpus run takes
its own immutable reference snapshot and caches decoded image fingerprints.
The configured model is checked on every server concurrently; any unavailable
server or missing model stops preflight before processing. This step does
not generate summaries or write artifacts.

In [ ]:
corpora = [(input_dir, discover_markdown(input_dir)) for input_dir in INPUT_DIRS]
for input_dir, md_files in corpora:
    print(f"{input_dir.name}: {len(md_files)} Markdown files")
print(f"Total Markdown files: {sum(len(files) for _, files in corpora)}")
ignored_image_snapshot = await object_utils._prepare_ignored_images(root=PROJ_ROOT)
print(f"Usable ignored-image references: {len(ignored_image_snapshot.references)}")
print(f"Reference signature: {ignored_image_snapshot.signature[:12]}")
print(
    f"Near-duplicate limits: aspect {object_utils._IGNORED_IMAGE_ASPECT_TOLERANCE:.0%}, "
    f"hash {object_utils._IGNORED_IMAGE_HASH_DISTANCE}/64 bits, "
    f"RGB difference {object_utils._IGNORED_IMAGE_RGB_DIFFERENCE:.0%}"
)
await ollama_client.check_models(SUMMARY_MODEL)
print(f"Providers ready on both/all configured servers: {SUMMARY_MODEL}")

## Rerun object summarization

Process 4.0 first, followed by 5.0, with bounded concurrent documents within each
corpus. Each successful document replaces its adjacent `_semantics.json` only
after its complete tree and document summary are ready. Every table row and
nested merged group is summarized bottom-up before the complete table. Failed
or cancelled documents keep their previous output; other documents continue.

Progress is atomically checkpointed in each corpus's `.object-summary-checkpoint.json`,
separately from natural-text checkpoints. With `RESUME=True`, unchanged successful
documents are checked for gaps, including after new summary features or object
schema versions are added. Complete outputs are skipped. Outputs missing table
rows, merged groups, list items, or object summaries are upgraded incrementally:
only missing summaries require model requests. Existing object, prose, section,
and document summaries and original Pandoc subtrees are preserved. Existing
parent summaries are not regenerated when descendants are added. Keep
`RESUME=True` across feature additions; use `RESUME=False` only for an explicitly
requested full rebuild. Successful upgrades atomically replace the output and
update its checkpoint signature. `incremental=True` identifies an upgraded
document; `filled_summaries` counts the newly filled object summaries. Failed or
cancelled upgrades retain the previous output and completed checkpoint so a
rerun can retry filling the gaps.

Changes to Markdown, cached-artifact signatures, output, model, language, summary
thresholds, ignored-image reference contents, or matching settings still cause
reprocessing. Reference additions and removals also invalidate checkpoints
automatically. Interrupted or failed documents retain unpublished work in an adjacent
`_summary-progress.json` file, written atomically after each completed outer
object and when workers stop. With `RESUME=True`, compatible partial progress
restores completed object counts and requests only missing object/descendant
summaries. Partial progress uses the same source/output signatures and model
settings checks; it never replaces published semantics or a completed checkpoint.
The partial file is removed after successful publication and checkpointing.
Interruptions from older runs that did not save partial work cannot be recovered. Keep
checkpoints alongside their corpora.

Progress bars show finished documents (including failures and resumed results)
and outer objects in currently active documents. Each active file also has its own
row combining an object progress bar with the file path, stage, queued and running
work, and elapsed time. Worker rows are reused for the next file and refresh every
250ms, including during final summaries. Per-file `sections done/total` counts
all nested sections and advances only after a section summary finishes.
The Documents bar also shows the aggregate section ratio for active files.
Section completion is separate from object completion and the final document
summary; interrupted traversals may summarize sections again. `last_doc` and `mean_doc` report
actual wall times of documents processed or upgraded in this run, excluding skipped documents;
concurrent document durations overlap. Request counts, in-flight requests, oldest
request age, and each active document's elapsed time remain visible while waiting.
The former `s/doc` was elapsed batch time divided by finished documents, so it
rose during long documents and was not an individual document latency. A table completes
after all descendant rows and groups; a fully ignored figure completes with an
empty summary and no model requests. Displays refresh every **250ms**, with a
final flush. Section and document-summary requests may follow object completion.

Interrupting cancels outstanding model work, awaits image-decoding cleanup, and
saves completed-document checkpoints. Synchronous Markdown/Pandoc parsing finishes
before cancellation can be observed.
Each worker publishes its document and starts the next pending file independently.
An object's `N/N` count does **not** mean the whole document is complete: prose,
parent sections, and the final document summary can still be pending. Per-document
status shows the final-summary stage plus `queued` and `running` work at the shared
concurrency limit (model requests and local image decoding). A `COMPLETED` message
confirms publication and checkpointing. Async client overlap does not guarantee
parallel inference on the Ollama server.

Per-server active requests appear with consistent color badges (S1 blue, S2 orange,
then green, purple, red, yellow). Each file row shows every server currently
processing its requests; its bar takes the color of the busiest server (ties use
server order), and returns to the default color when no server is active.
Server URLs and request counts remain visible alongside the colors. Server-list changes preserve
compatible completed results; use `RESUME=False` to force rebuilding. A failed request fails its document and releases the
server slot; other documents continue. Requests are not automatically retried.


In [ ]:
from html import escape
from IPython.display import HTML, display

ollama_client.reset_metrics()
batch_start = perf_counter()
results = []
completed_by_file = {}
for input_dir, _ in corpora:
    checkpoint_file = input_dir / CHECKPOINT_NAME
    if checkpoint_file.is_file():
        saved_checkpoint = json.loads(checkpoint_file.read_text(encoding="utf-8"))
        for file, entry in saved_checkpoint.get("entries", {}).items():
            if entry.get("row", {}).get("status") == "completed":
                completed_by_file[str(Path(file).resolve())] = dict(entry["row"], file=file)


def show_completed_files(rows):
    global completed_display, completed_display_files, completed_files
    for row in rows:
        completed_by_file[str(Path(row["file"]).resolve())] = row
    paths = tuple(sorted(completed_by_file))
    completed_files = [completed_by_file[file] for file in paths]
    if paths == completed_display_files:
        return
    content = HTML(
        f"<div><strong>Completed files, including past runs ({len(paths)}):</strong>"
        + "<pre style='white-space:pre-wrap'>"
        + escape("\n".join(paths) if paths else "No completed files yet.")
        + "</pre></div>"
    )
    if completed_display is None:
        # Called after the file bars are created: reserve the output below them.
        completed_display = display(content, display_id=True)
    else:
        completed_display.update(content)
    completed_display_files = paths


try:
    for input_dir, md_files in corpora:
        print(f"Processing corpus: {input_dir.name}")
        completed_display = None
        completed_display_files = None
        corpus_results = await summarize_object_documents(
            md_files, ollama_client, SUMMARY_MODEL, corpus=input_dir.name,
            concurrency=DOCUMENT_CONCURRENCY, progress_interval=0.25,
            checkpoint_path=input_dir / CHECKPOINT_NAME,
            resume=RESUME, checkpoint_host=OLLAMA_HOSTS,
            lang=LANG, leaf_min_len=LEAF_MIN_LEN, min_len=MIN_LEN,
            completed_callback=show_completed_files,
        )
        results.extend(corpus_results)
finally:
    batch_elapsed_seconds = perf_counter() - batch_start
    batch_metrics = ollama_client.metrics.snapshot(wall_seconds=batch_elapsed_seconds)
    server_metrics = ollama_client.server_snapshots(wall_seconds=batch_elapsed_seconds)


## Results

Review failures before rerunning. Unchanged successful documents are marked
`resumed=True`; their timings and request metrics come from their saved run.
Batch metrics describe only requests made during this run. Output paths identify
the complete recursive JSON trees.

`object_count` counts outer figures, tables, and lists. Rows and nested merged
groups remain inside their table and do not increase this count.
`completed_objects` includes intentionally ignored figures and completed tables,
even if a later parent summary fails. Use the saved-output preview below to inspect
individual row and list item summaries, recursive children, and ignored-image markers.

In [ ]:
results_df = pd.DataFrame(results)
display(results_df)
if not results_df.empty:
    print(results_df.groupby(["corpus", "status"]).size().to_string())
    print(f"Summed document processing time: {results_df['elapsed_seconds'].sum():.3f}s")
    failures = results_df.loc[results_df["status"] == "failed"]
    if not failures.empty:
        print("Failed documents:")
        display(failures[["corpus", "file", "error"]])
print(f"Batch wall time: {batch_elapsed_seconds:.3f}s")
display(pd.DataFrame([batch_metrics]))
print("Per-server inference metrics:")
display(pd.DataFrame(server_metrics))


## Inspect saved objects, rows, and ignored figures

Read up to `PREVIEW_OUTPUTS` successful output files, including resumed results.
The view follows outer objects and their recursive `subnodes` once, showing
parent IDs, depth, row ranges, header rows, summary status, source lines, and
ignored-image counts. It checks row-wrapper alignment and accepts empty summaries
only for explicitly ignored figures. This cell makes no model requests and writes
no files. Set `PREVIEW_OUTPUTS = 0` to skip reading saved trees.

In [ ]:
def summary_records(tree):
    """Read saved semantic wrappers once, including recursive table and list descendants."""
    from ribosome.core.dom.summary.object import _object_nodes

    records = []

    def visit(node, parent_id=None, depth=0):
        obj = node["object"]
        summary = obj.get("summary")
        ignored = obj.get("summary_ignored", False)
        valid_summary = isinstance(summary, str) and (
            bool(summary.strip()) if not ignored else obj["type"] == "figure" and summary == ""
        )
        if not valid_summary or node.get("s") != summary:
            raise ValueError(f"Invalid summary for {obj['type']} object {obj['id']}")
        children = obj.get("children", [])
        subnodes = node.get("subnodes", [])
        if [child["object"]["id"] for child in subnodes] != [child["id"] for child in children]:
            raise ValueError(f"Object hierarchy mismatch for object {obj['id']}")
        records.append({
            "id": str(obj["id"]), "parent_id": parent_id, "depth": depth,
            "type": obj["type"], "row_start": obj.get("row_start", obj.get("row_index")),
            "row_end": obj.get("row_end", obj.get("row_index")),
            "is_header": obj.get("is_header", False),
            "item_index": obj.get("item_index"), "ordinal": obj.get("ordinal"),
            "list_type": obj.get("list_type"), "list_id": obj.get("list_id"),
            "status": "ignored" if ignored else "summarized",
            "ignored_images": sum(bool(image.get("ignored_reference")) for image in obj.get("images", [])),
            "line_start": obj["line_start"], "line_end": obj["line_end"], "summary": summary,
        })
        for child in subnodes:
            visit(child, str(obj["id"]), depth + 1)

    for node in _object_nodes(tree):
        visit(node)
    return records


preview_outputs = [row["output"] for row in results if row["status"] == "completed"][:PREVIEW_OUTPUTS]
for output in preview_outputs:
    print(f"Saved object, row and list item summaries: {output}")
    saved_tree = json.loads(Path(output).read_text(encoding="utf-8"))
    preview_df = pd.DataFrame(summary_records(saved_tree))
    if not preview_df.empty:
        print(preview_df.groupby(["type", "status"]).size().to_string())
    display(preview_df)


## Performance and reruns

Figures require one image request per retained image plus a combined summary
request. Fully ignored figures require neither request. Each physical table row,
merged subgroup, and complete table has its own summary request; deeply nested
or large tables can therefore take longer than an outer-object count suggests.
Parent and document summaries also contribute to runtime.

Compare client latency, Ollama prompt/generation timings, and peak requests before
adjusting concurrency. Client request overlap is not the same as server inference
parallelism. Change limits before importing the object module and restart the
kernel when prompted.

Keep `RESUME=True` after interruptions and when new summarizer features are
added. Schema-version changes preserve completed checkpoint entries instead of
repeating model requests. Completed outputs are checked for missing descendants and
upgraded using the latest recursion, with existing summaries preserved. Fully
complete outputs are skipped without model requests or rewriting the output. Changes to the ignored-image reference set and matching
settings still automatically rebuild affected corpus checkpoints. Use
`RESUME=False` only for an explicitly requested full rebuild; it replaces the
object checkpoint without altering Markdown or AST caches.

The natural-text benchmark utility measures a different pipeline and is not used
here. No corpus processing is performed during notebook creation or offline validation.
Results include aggregate and per-server metrics. To release HTTP resources after
finishing, run `await ollama_client.aclose()`; rerun setup before another batch.
